# 02. 내 연주가 눈에 보이다 — **확장형 · 시각 차원**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장형 ✦ : NB02 (시각 차원) ← 여기 + NB03 (음악 차원)
협업형   : NB04 (AI와 대화)
통합     : NB05 (무대)
```

**확장**이란 연주가 단일한 소리 이벤트에 머무르지 않고 **새로운 차원으로 번지는 것**입니다.
여기서는 연주를 **시각 세계로 확장**합니다.

## 핵심: 매핑을 직접 디자인

이 노트북의 요점은 "AI가 만든 비주얼을 감상"하는 것이 아니라, **내가 음악 특성 → 시각 파라미터 연결을 결정**하는 것입니다.

같은 Satie 연주라도:
- `energy → sphere_size` 매핑: 다이내믹이 오브젝트 크기를 조종
- `harmonic_tension → bg_saturation` 매핑: 화성이 공기 채도를 조종

→ 질적으로 다른 시각 세계가 됩니다. **확장의 결과를 학생이 주도**합니다.

**입력**: `artifacts/midi/*.mid`, `artifacts/analysis/*.json` (NB01 출력)  
**출력**: `artifacts/cues/*.json` → NB04·NB05에서 사용

In [ ]:
!pip install -q basic-pitch librosa pretty_midi numpy matplotlib

import warnings
warnings.filterwarnings('ignore')
print('✅ 설치 완료!')

---
## 1. NB01 산출물 로드

채보된 MIDI + 오디오 원본을 함께 사용합니다.

> NB01을 실행하지 않았더라도 괜찮습니다 — MIDI 파일이 없으면 자동으로 Basic Pitch 채보를 수행합니다.

In [ ]:
from pathlib import Path

ASSETS = Path('assets')
ARTIFACTS = Path('artifacts')
(ARTIFACTS / 'midi').mkdir(parents=True, exist_ok=True)
(ARTIFACTS / 'cues').mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'title': 'Satie — Gymnopédie No.1',
        'audio': ASSETS / 'satie_gymnopedie_no1.wav',
        'midi': ARTIFACTS / 'midi' / 'satie.mid',
    },
    'prokofiev': {
        'title': 'Prokofiev — Toccata Op.11',
        'audio': ASSETS / 'prokofiev_toccata_op11.wav',
        'midi': ARTIFACTS / 'midi' / 'prokofiev.mid',
    },
}

need_transcription = [k for k, p in pieces.items() if not p['midi'].exists()]
if need_transcription:
    print(f"⚠️ NB01 산출물 없음 → Basic Pitch 채보를 자동 실행합니다: {need_transcription}")
    from basic_pitch.inference import predict
    for key in need_transcription:
        p = pieces[key]
        print(f"  🔄 {p['title']} 채보 중...")
        _, midi_data, note_events = predict(str(p['audio']))
        midi_data.write(str(p['midi']))
        print(f"  ✅ {len(note_events)}개 음표 → {p['midi']}")

for key, p in pieces.items():
    print(f"✅ {p['title']}: {p['midi']}")

---
## 2. cue 추출 (피아노 전용 채널 포함)

프레임 단위(24fps)로 음악 특성을 추출합니다. **기본 cue 5개 + 피아노 전용 cue 3개 + 펄스**:

| cue | 의미 | Satie 예상 | Prokofiev 예상 |
|---|---|---|---|
| `energy` | RMS 음량 | 낮음 (flat) | 높음 (격동) |
| `density` | **동시 울리는** 음표 수 (지속음 포함) | 중간 (왈츠 베이스 지속) | 중간 (짧은 스타카토) |
| `onset_density` ★ | 1초 윈도우 **새 음표 수** | 낮음 (~2/s) | 높음 (~10/s) |
| `register` | 평균 pitch | 중간 | 급변동 |
| `brightness` | spectral centroid | 낮음 | 높음 |
| `attack` | 프레임당 새 onset | 희소 | 밀집 |
| `register_spread` ★ | pitch 분산 (그 순간 음역 폭) | 작음 | 큼 |
| `harmonic_tension` ★ | 불협화도 (m2·tritone 비율) | 낮음 | 높음 |
| `velocity_variance` ★ | velocity 분산 | 거의 0 | 높음 |
| `pulse` | onset peak (beat 대용) | 왈츠 리듬 | 16분 펄스 |

**주의**: `density`와 `onset_density`의 차이
- `density`: "지금 이 순간 몇 음이 울리고 있나" (**지속음 가중**). 왈츠 반주처럼 긴 음을 누르고 있으면 높아짐.
- `onset_density`: "1초당 몇 번 새 음이 시작되나" (**직관적 텍스처 속도**). 스타카토가 많으면 높아짐.

Satie는 density는 중간, onset_density는 낮음. Prokofiev는 density는 중간, onset_density는 높음.

**Global normalization**: 두 곡을 같은 절대 스케일로 정규화해서 **대비를 보존**합니다.

**Basic Pitch 한계**: 채보는 정확하지만 **velocity 추정이 평균값으로 수렴**하는 경향. `velocity_variance`와 `velocity_std`(NB01)는 참고용입니다.

In [ ]:
import numpy as np
import librosa
import pretty_midi

# Global reference scales — 두 곡의 절대 대비를 보존
GLOBAL_SCALES = {
    'energy': 0.4,               # RMS 0~0.4를 0~1로
    'density': 12.0,             # 동시 12음을 1.0으로
    'onset_density': 15.0,       # 초당 15 onset을 1.0으로
    'attack': 6.0,               # 프레임당 6 onset을 1.0으로
    'register_spread': 60.0,     # 5옥타브 폭을 1.0으로
    'velocity_variance': 40.0,   # velocity σ=40을 1.0으로
}

DISSONANT_INTERVALS = {1, 2, 6, 10, 11}  # 반음 단위: m2, M2, tritone, m7, M7

def extract_cues(audio_path: Path, midi_path: Path, fps: int = 24) -> dict:
    y, sr = librosa.load(str(audio_path), sr=22050)
    duration = len(y) / sr
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = sorted([n for inst in pm.instruments for n in inst.notes], key=lambda n: n.start)

    frame_times = np.arange(0, duration, 1 / fps)
    n_frames = len(frame_times)

    # 오디오 기반
    rms = librosa.feature.rms(y=y)[0]
    rms_times = librosa.times_like(rms, sr=sr)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    centroid_times = librosa.times_like(centroid, sr=sr)

    energy = np.interp(frame_times, rms_times, rms)
    brightness_raw = np.interp(frame_times, centroid_times, centroid)
    brightness = brightness_raw / 5000.0  # ~5kHz를 1.0으로

    # MIDI 기반 — 프레임별 활성 노트
    density = np.zeros(n_frames)
    register_sum = np.zeros(n_frames)
    active_counts = np.zeros(n_frames)
    attack_arr = np.zeros(n_frames)
    pitch_min = np.full(n_frames, np.nan)
    pitch_max = np.full(n_frames, np.nan)
    velocity_var = np.zeros(n_frames)
    dissonance = np.zeros(n_frames)

    for i in range(n_frames):
        t = frame_times[i]
        active = [n for n in notes if n.start <= t < n.end]
        if active:
            pitches = [n.pitch for n in active]
            vels = [n.velocity for n in active]
            density[i] = len(active)
            register_sum[i] = sum(pitches)
            active_counts[i] = len(active)
            pitch_min[i] = min(pitches)
            pitch_max[i] = max(pitches)
            velocity_var[i] = np.std(vels) if len(vels) > 1 else 0
            # harmonic tension: 불협 간격 비율
            if len(pitches) >= 2:
                intervals = []
                for a in range(len(pitches)):
                    for b in range(a+1, len(pitches)):
                        intervals.append(abs(pitches[a] - pitches[b]) % 12)
                diss_count = sum(1 for iv in intervals if iv in DISSONANT_INTERVALS)
                dissonance[i] = diss_count / len(intervals)

    # attack: 프레임 내 새 onset
    for n in notes:
        idx = int(np.clip(n.start * fps, 0, n_frames - 1))
        attack_arr[idx] += 1

    # onset_density: 1초 sliding window 내 새 onset 수 (notes/sec)
    window_half = fps // 2  # 0.5초 반폭
    onset_density = np.zeros(n_frames)
    for i in range(n_frames):
        lo = max(0, i - window_half)
        hi = min(n_frames, i + window_half + 1)
        onset_density[i] = attack_arr[lo:hi].sum()  # ≈ notes in 1 second

    register_raw = np.where(active_counts > 0, register_sum / np.maximum(active_counts, 1), 60)
    register = (register_raw - 21) / (108 - 21)  # MIDI 21(A0) ~ 108(C8)
    register_spread = np.where(
        ~np.isnan(pitch_max), pitch_max - np.nan_to_num(pitch_min), 0
    )

    # Global normalization
    def gnorm(arr, ref): return np.clip(arr / ref, 0, 1)

    # pulse: onset density local peaks (beat_track 대용, 루바토에 강건)
    pulse = np.zeros(n_frames)
    window = max(1, fps // 4)  # 0.25초 윈도우
    for i in range(window, n_frames - window):
        local = attack_arr[i-window:i+window+1]
        if attack_arr[i] > 0 and attack_arr[i] == local.max():
            pulse[i] = 1

    frames = []
    for i, t in enumerate(frame_times):
        frames.append({
            't': round(float(t), 4),
            'energy': round(float(gnorm(energy[i], GLOBAL_SCALES['energy'])), 4),
            'density': round(float(gnorm(density[i], GLOBAL_SCALES['density'])), 4),
            'onset_density': round(float(gnorm(onset_density[i], GLOBAL_SCALES['onset_density'])), 4),
            'register': round(float(np.clip(register[i], 0, 1)), 4),
            'brightness': round(float(np.clip(brightness[i], 0, 1)), 4),
            'attack': round(float(gnorm(attack_arr[i], GLOBAL_SCALES['attack'])), 4),
            'register_spread': round(float(gnorm(register_spread[i], GLOBAL_SCALES['register_spread'])), 4),
            'harmonic_tension': round(float(dissonance[i]), 4),
            'velocity_variance': round(float(gnorm(velocity_var[i], GLOBAL_SCALES['velocity_variance'])), 4),
            'pulse': int(pulse[i]),
        })

    return {
        'duration': round(float(duration), 3),
        'fps': fps,
        'normalization': 'global',
        'scales': GLOBAL_SCALES,
        'frames': frames,
    }


import json
for key, p in pieces.items():
    print(f"🔄 {p['title']} cue 추출 중...")
    cues = extract_cues(p['audio'], p['midi'])
    out_path = ARTIFACTS / 'cues' / f'{key}_visual_cues.json'
    out_path.write_text(json.dumps({'piece': p['title'], 'audio_file': str(p['audio']), **cues}, ensure_ascii=False))
    p['cues'] = cues
    # 요약 출력
    f = cues['frames']
    for ch in ['density', 'onset_density', 'harmonic_tension']:
        vals = [fr[ch] for fr in f]
        print(f"   {ch:16s}: mean={np.mean(vals):.3f}  max={max(vals):.3f}")
    print(f"   ✅ {len(f)} 프레임 → {out_path}")

---
## 3. 두 곡 cue 비교

Global normalization 덕분에 두 곡의 **절대적 대비**가 그래프로 드러납니다.

In [ ]:
import matplotlib.pyplot as plt

cue_channels = ['energy', 'onset_density', 'density', 'register_spread', 'harmonic_tension']
fig, axes = plt.subplots(len(cue_channels), 1, figsize=(14, 12), sharex=False)

for ax, cue_name in zip(axes, cue_channels):
    for key, color in [('satie', '#4f46e5'), ('prokofiev', '#dc2626')]:
        frames = pieces[key]['cues']['frames']
        t = [f['t'] for f in frames]
        v = [f[cue_name] for f in frames]
        ax.plot(t, v, color=color, alpha=0.7, label=pieces[key]['title'], linewidth=1)
    ax.set_ylabel(cue_name)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('time (s)')
axes[0].set_title('cue 비교 (global normalized 0~1)')
plt.tight_layout()
plt.show()

print('💡 관찰:')
print('   - onset_density: Prokofiev가 명확히 높음 (타격 밀도)')
print('   - density: 의외로 비슷하거나 역전 (지속음 가중)')
print('   - register_spread: Prokofiev가 프레임별 음역 폭 큼')
print('   - harmonic_tension: 두 곡 모두 불협 순간 존재')

---
## 4. 매핑 설계 — 학생 주체성의 핵심

아래 딕셔너리에서 **왼쪽(시각 파라미터)** 에 **오른쪽(음악 cue)** 을 연결합니다.

값만 바꿔서 재실행하면 같은 연주가 다른 세계가 됩니다.

### 프리셋
- **서정적**: 다이내믹보다 화성·음색 중심
- **역동적**: energy·attack이 카메라·크기를 움직임
- **미니멀**: 대부분 고정, pulse만 반응
- **custom**: 자유 매핑

### 시각 파라미터
- `sphere_scale`: 중심 구체 크기
- `sphere_hue`: 구체 색상 (0-360)
- `bar_height`: 34개 막대 높이
- `particle_count`: 파티클 개수 강도
- `bg_saturation`: 배경 채도
- `camera_shake`: 카메라 흔들림
- `rotation_speed`: 회전 속도
- `torus_pulse`: beat 링 강도
- `bloom`: 발광 효과

### 연결 가능한 cue (10개)
`energy`, `density`, `onset_density`, `register`, `brightness`, `attack`, `register_spread`, `harmonic_tension`, `velocity_variance`, `pulse`

In [ ]:
PRESETS = {
    '서정적': {
        'sphere_scale':    'brightness',
        'sphere_hue':      'register',
        'bar_height':      'density',
        'particle_count':  'harmonic_tension',
        'bg_saturation':   'energy',
        'camera_shake':    'velocity_variance',
        'rotation_speed':  'register_spread',
        'torus_pulse':     'pulse',
        'bloom':           'energy',
    },
    '역동적': {
        'sphere_scale':    'energy',
        'sphere_hue':      'harmonic_tension',
        'bar_height':      'onset_density',
        'particle_count':  'attack',
        'bg_saturation':   'brightness',
        'camera_shake':    'energy',
        'rotation_speed':  'onset_density',
        'torus_pulse':     'pulse',
        'bloom':           'attack',
    },
    '미니멀': {
        'sphere_scale':    'density',
        'sphere_hue':      'register',
        'bar_height':      'pulse',
        'particle_count':  'attack',
        'bg_saturation':   'harmonic_tension',
        'camera_shake':    'pulse',
        'rotation_speed':  'pulse',
        'torus_pulse':     'pulse',
        'bloom':           'brightness',
    },
}

# ★ 여기를 직접 수정하세요 ★
my_mapping = PRESETS['역동적'].copy()
# 예: my_mapping['sphere_scale'] = 'harmonic_tension'

print('현재 매핑:')
for k, v in my_mapping.items():
    print(f'   {k:18s} ← {v}')

---
## 5. p5.js WEBGL 렌더링

브라우저에서 재생 버튼을 누르면 연주와 함께 3D 장면이 반응합니다.

**곡 선택**: `render_piece` 변수를 `'satie'` 또는 `'prokofiev'`로 바꿔 재실행.

In [ ]:
import base64
from uuid import uuid4
from IPython.display import HTML, display

# ★ 렌더링할 곡 선택 ★
render_piece = 'prokofiev'  # 'satie' or 'prokofiev'

p = pieces[render_piece]
audio_bytes = p['audio'].read_bytes()
audio_b64 = base64.b64encode(audio_bytes).decode('ascii')
cues_data = {'audio_file': str(p['audio']), **p['cues']}
cues_json = json.dumps(cues_data, ensure_ascii=False)
mapping_json = json.dumps(my_mapping)
audio_mime = 'audio/wav'
cid = f"av-{uuid4().hex}"

html = f"""
<div id="{cid}" style="font-family:'IBM Plex Sans KR',sans-serif;color:#e5e7eb;background:linear-gradient(135deg,#050816,#101828);border:1px solid rgba(255,255,255,0.08);border-radius:24px;padding:20px;">
  <div style="display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap;margin-bottom:14px;">
    <div>
      <div style="font-size:24px;font-weight:800;">{p['title']}</div>
      <div style="font-size:13px;color:#94a3b8;margin-top:4px;">매핑: {list(my_mapping.values())[:3]}...</div>
    </div>
    <button id="{cid}-btn" style="border:none;border-radius:999px;padding:12px 18px;background:#f59e0b;color:#111827;font-size:14px;font-weight:700;cursor:pointer;">재생</button>
  </div>
  <div id="{cid}-canvas" style="min-height:540px;border-radius:20px;overflow:hidden;background:#020617;"></div>
  <div id="{cid}-readout" style="display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:8px;margin-top:12px;font-size:12px;color:#cbd5e1;"></div>
</div>
<script src="https://cdn.jsdelivr.net/npm/p5@1.11.1/lib/p5.min.js"></script>
<script>
(() => {{
  const data = {cues_json};
  const mapping = {mapping_json};
  const frames = data.frames;
  const audio = new Audio('data:{audio_mime};base64,{audio_b64}');
  const btn = document.getElementById('{cid}-btn');
  const readout = document.getElementById('{cid}-readout');
  const host = document.getElementById('{cid}-canvas');

  function fAt(t) {{ const i = Math.min(Math.floor(t * data.fps), frames.length - 1); return frames[i]; }}
  function M(param, f) {{ const cue = mapping[param]; return f && cue in f ? f[cue] : 0; }}

  function renderReadout(f) {{
    if (!f) return;
    const keys = ['energy','density','register_spread','harmonic_tension','pulse'];
    readout.innerHTML = keys.map(k => `<div style="border:1px solid rgba(255,255,255,0.10);border-radius:12px;padding:8px;background:rgba(15,23,42,0.55);"><div style="font-size:10px;letter-spacing:0.12em;color:#f59e0b;">${{k}}</div><div style="font-size:15px;font-weight:700;color:#f8fafc;">${{(+f[k]).toFixed(2)}}</div></div>`).join('');
  }}

  btn.addEventListener('click', async () => {{
    if (audio.paused) {{ await audio.play(); btn.textContent = '일시정지'; }}
    else {{ audio.pause(); btn.textContent = '재생'; }}
  }});
  audio.addEventListener('ended', () => {{ btn.textContent = '다시 재생'; }});

  new p5((p) => {{
    const N_PARTICLES = 120;
    const particles = Array.from({{length: N_PARTICLES}}, (_, i) => ({{
      angle: (Math.PI * 2 * i) / N_PARTICLES, depth: p.random(-220, 220), seed: p.random(1000),
    }}));
    p.setup = () => {{
      const c = p.createCanvas(host.clientWidth || 900, 540, p.WEBGL); c.parent(host);
      p.colorMode(p.HSB, 360, 100, 100, 100); p.noStroke(); renderReadout(fAt(0));
    }};
    p.windowResized = () => p.resizeCanvas(host.clientWidth || 900, 540);
    p.draw = () => {{
      const now = audio.currentTime || 0;
      const f = fAt(now) || frames[0]; if (!f) return;
      renderReadout(f);
      const sphereScale = 0.5 + 2.5 * M('sphere_scale', f);
      const hue = (M('sphere_hue', f) * 360) % 360;
      const barH = M('bar_height', f);
      const particleDensity = M('particle_count', f);
      const saturation = 20 + 60 * M('bg_saturation', f);
      const shake = M('camera_shake', f);
      const rot = M('rotation_speed', f);
      const torus = M('torus_pulse', f);
      const bloom = M('bloom', f);
      const orbit = now * (0.3 + rot * 1.2);
      p.background(228, saturation, 8 + bloom * 12);
      p.ambientLight(220, 20, 35);
      p.pointLight(hue, 70, 100, 0, -180, 260);
      p.pointLight((hue + 140) % 360, 60, 100, 180, 120, 120);
      // camera shake
      p.push();
      p.translate((p.random() - 0.5) * shake * 60, (p.random() - 0.5) * shake * 60, 0);
      // bars
      p.push(); p.rotateY(orbit); p.rotateX(0.75);
      for (let i = 0; i < 34; i++) {{
        const x = p.map(i, 0, 33, -260, 260);
        const h = 35 + barH * 250 * (0.5 + 0.5 * Math.sin(now * 1.2 + i * 0.3));
        p.push(); p.translate(x, -20, -120 + i * 6);
        p.fill((hue + i * 3.5) % 360, 60, 95, 70);
        p.box(12, Math.abs(h), 14);
        p.pop();
      }} p.pop();
      // central sphere
      p.push(); p.rotateY(-orbit * 0.5);
      p.fill((hue + 40) % 360, 40, 100, 18 + bloom * 24);
      p.sphere(80 * sphereScale, 48, 32);
      p.pop();
      // particles
      p.push(); p.blendMode(p.ADD);
      const visibleN = Math.floor(N_PARTICLES * (0.1 + 0.9 * particleDensity));
      for (let i = 0; i < visibleN; i++) {{
        const pt = particles[i];
        const radius = 150 + 140 * particleDensity;
        const theta = pt.angle + orbit * 0.8;
        const x = Math.cos(theta) * radius;
        const y = Math.sin(theta * 1.8 + pt.seed) * (70 + 90 * M('sphere_scale', f));
        const z = pt.depth + Math.sin(now + pt.seed) * 40;
        p.push(); p.translate(x, y, z);
        p.fill((hue + 120 + i) % 360, 35, 100, 55);
        p.sphere(2 + 7 * bloom, 8, 8);
        p.pop();
      }} p.blendMode(p.BLEND); p.pop();
      // torus on pulse
      if (torus > 0) {{
        p.push(); p.rotateX(Math.PI / 2);
        p.noFill(); p.stroke((hue + 180) % 360, 50, 100, 65); p.strokeWeight(3);
        p.torus(170 + 60 * bloom, 7 + 12 * torus, 40, 10); p.noStroke();
        p.pop();
      }}
      p.pop();
    }};
  }});
}})();
</script>
"""
display(HTML(html))

---
## 6. 매핑 저장 (NB05에서 사용)

In [ ]:
mapping_out = ARTIFACTS / 'cues' / 'my_mapping.json'
mapping_out.write_text(json.dumps({'presets': PRESETS, 'current': my_mapping}, ensure_ascii=False, indent=2))
print(f'✅ 매핑 저장: {mapping_out}')
print()
print('💡 다른 매핑으로 바꿔서 재실행해보세요.')
print('   예: my_mapping["sphere_scale"] = "harmonic_tension"')
print('   같은 연주가 다른 성격의 시각 세계가 됩니다.')

---
## 다음 단계

**→ NB04 (`04_collaborate.ipynb`)**: AI가 내 연주를 실시간으로 추적합니다 — matchmaker 기반 악보 추적(score following).